In [8]:
print('ds')

ds


## Step 0. 검증 목적과 주의사항

### 목적
- 소음 세그먼트 점수의 품질을 빠르게 검증합니다.
- 핵심: `VIF(다중공선성)` + `재현성 상관계수` + `순위 변동`

### 주의사항
1. `총점`은 VIF 입력에서 제외 (종속적으로 만들어진 값이므로 왜곡됨)
2. 같은 의미의 중복 변수(예: 원점수와 정규화점수 동시 사용)는 VIF에서 제외
3. VIF는 연도별로 따로 계산 권장 (분포가 연도마다 다를 수 있음)
4. 재현성은 Pearson + Spearman 둘 다 확인
5. 결측치 처리 방식(dropna)을 로그로 남김


In [9]:
%pip install statsmodels

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.outliers_influence import variance_inflation_factor


## Step 1. 입력 CSV 로드

### 파일
- 아래 `input_csv`를 실제 파일로 바꿔주세요.
- 예: `noise_vibration_final_long_no_rank.csv` 또는 최종 점수 파일


In [ ]:


df = pd.read_csv("소음최종점수_2022_2024.csv", encoding="utf-8-sig")
print("shape:", df.shape)
print("columns:", df.columns.tolist())
df.head()


shape: (25, 16)
columns: ['자치구', '민원스트레스점수_2022', '민원스트레스점수_2023', '민원스트레스점수_2024', '소음스트레스점수_2022', '소음스트레스점수_2023', '소음스트레스점수_2024', '진동스트레스점수_2022', '진동스트레스점수_2023', '진동스트레스점수_2024', '소음원점수_2022', '소음원점수_2023', '소음원점수_2024', '소음총점수_2022', '소음총점수_2023', '소음총점수_2024']


,자치구,민원스트레스점수_2022,민원스트레스점수_2023,민원스트레스점수_2024,소음스트레스점수_2022,소음스트레스점수_2023,소음스트레스점수_2024,진동스트레스점수_2022,진동스트레스점수_2023,진동스트레스점수_2024,소음원점수_2022,소음원점수_2023,소음원점수_2024,소음총점수_2022,소음총점수_2023,소음총점수_2024
0,종로구,6.09,6.98,6.52,3.36,3.20,3.52,3.96,4.16,4.73,13.41,14.34,14.77,13.97,14.94,15.39
1,중구,8.00,4.44,4.25,3.03,2.13,2.34,2.63,2.96,4.37,13.66,9.53,10.96,14.23,9.93,11.42
2,용산구,4.97,6.67,5.83,4.45,6.69,3.20,3.34,4.02,4.14,12.76,17.38,13.17,13.29,18.10,13.72
3,성동구,3.49,4.33,4.16,6.29,8.00,6.66,3.87,4.62,5.09,13.65,16.95,15.91,14.22,17.66,16.57
4,광진구,2.13,1.90,1.78,4.94,3.40,2.68,4.25,4.73,4.82,11.32,10.03,9.28,11.79,10.45,9.67


----------------------------------------

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr

df = pd.read_csv("noise_rank_compare.csv", encoding="utf-8-sig")

print(df.columns.tolist())
display(df.head())


['자치구', '총점_orig_2022', '총점_interp_2022', '순위_orig_2022', '순위_interp_2022', '점수차(orig-interp)_2022', '순위차(orig-interp)_2022', '절대순위차_2022', '총점_orig_2023', '총점_interp_2023', '순위_orig_2023', '순위_interp_2023', '점수차(orig-interp)_2023', '순위차(orig-interp)_2023', '절대순위차_2023', '총점_orig_2024', '총점_interp_2024', '순위_orig_2024', '순위_interp_2024', '점수차(orig-interp)_2024', '순위차(orig-interp)_2024', '절대순위차_2024']


,자치구,총점_orig_2022,총점_interp_2022,순위_orig_2022,순위_interp_2022,점수차(orig-interp)_2022,순위차(orig-interp)_2022,절대순위차_2022,총점_orig_2023,총점_interp_2023,...,점수차(orig-interp)_2023,순위차(orig-interp)_2023,절대순위차_2023,총점_orig_2024,총점_interp_2024,순위_orig_2024,순위_interp_2024,점수차(orig-interp)_2024,순위차(orig-interp)_2024,절대순위차_2024
0,종로구,13.965278,13.951389,10.0,9.0,0.013889,1.0,1.0,15.628472,15.281250,...,0.347222,-1.0,1.0,15.347222,15.201389,6.0,6.0,0.145833,0.0,0.0
1,중구,14.392361,16.031250,6.0,2.0,-1.638889,4.0,4.0,10.347222,11.142361,...,-0.795139,1.0,1.0,11.468750,11.468750,13.0,13.0,0.000000,0.0,0.0
2,용산구,13.295139,13.843750,11.0,11.0,-0.548611,0.0,0.0,17.760417,14.777778,...,2.982639,-6.0,6.0,13.750000,13.750000,9.0,9.0,0.000000,0.0,0.0
3,성동구,14.138889,14.527778,9.0,6.0,-0.388889,3.0,3.0,17.659722,15.479167,...,2.180556,-1.0,1.0,16.486111,16.434028,3.0,3.0,0.052083,0.0,0.0
4,광진구,11.663194,11.295139,14.0,16.0,0.368056,-2.0,2.0,10.621528,9.951389,...,0.670139,-4.0,4.0,9.565972,9.565972,21.0,21.0,0.000000,0.0,0.0


In [14]:
# 자동으로 원본/보간 총점 컬럼 찾기 (예: ...orig..., ...interp...)
orig_cols = [c for c in df.columns if "orig" in c.lower() and "점수" in c.lower() or "orig" in c.lower() and "score" in c.lower() or "orig" in c.lower() and "총점" in c]
interp_cols = [c for c in df.columns if "interp" in c.lower() and "점수" in c.lower() or "interp" in c.lower() and "score" in c.lower() or "interp" in c.lower() and "총점" in c]

print("orig 후보:", orig_cols)
print("interp 후보:", interp_cols)


orig 후보: ['총점_orig_2022', '점수차(orig-interp)_2022', '총점_orig_2023', '점수차(orig-interp)_2023', '총점_orig_2024', '점수차(orig-interp)_2024']
interp 후보: ['총점_interp_2022', '점수차(orig-interp)_2022', '총점_interp_2023', '점수차(orig-interp)_2023', '총점_interp_2024', '점수차(orig-interp)_2024']


In [15]:
# 연도별로 직접 지정 (컬럼명 보고 맞게 수정)
score_pairs = {
    2022: ("총점_orig_2022", "총점_interp_2022"),
    2023: ("총점_orig_2023", "총점_interp_2023"),
    2024: ("총점_orig_2024", "총점_interp_2024"),
}

rows = []
for y, (c1, c2) in score_pairs.items():
    a = pd.to_numeric(df[c1], errors="coerce")
    b = pd.to_numeric(df[c2], errors="coerce")
    m = a.notna() & b.notna()

    pr, pp = pearsonr(a[m], b[m])
    sr, sp = spearmanr(a[m], b[m])

    rows.append({
        "연도": y,
        "표본수": int(m.sum()),
        "Pearson_r": round(pr, 6),
        "Spearman_rho": round(sr, 6),
        "평균점수차(orig-interp)": round((a[m] - b[m]).mean(), 6),
        "최대절대점수차": round((a[m] - b[m]).abs().max(), 6),
    })

repro = pd.DataFrame(rows)
display(repro)


,연도,표본수,Pearson_r,Spearman_rho,평균점수차(orig-interp),최대절대점수차
0,2022,25,0.969568,0.936923,-0.260000,1.638889
1,2023,25,0.931266,0.940000,-0.256944,2.982639
2,2024,25,0.999848,1.000000,0.010417,0.166667


In [16]:
# 순위 변동 요약
rank_rows = []
for y, (c1, c2) in score_pairs.items():
    t = df[[c1, c2]].copy()
    t[c1] = pd.to_numeric(t[c1], errors="coerce")
    t[c2] = pd.to_numeric(t[c2], errors="coerce")
    t = t.dropna()

    r1 = t[c1].rank(ascending=False, method="min")
    r2 = t[c2].rank(ascending=False, method="min")
    shift = (r1 - r2).abs()

    rank_rows.append({
        "연도": y,
        "평균절대순위변화": round(float(shift.mean()), 4),
        "최대순위변화": round(float(shift.max()), 4),
        "순위변화발생개수": int((shift > 0).sum()),
    })

rank_summary = pd.DataFrame(rank_rows)
display(rank_summary)


,연도,평균절대순위변화,최대순위변화,순위변화발생개수
0,2022,2.00,6.0,21
1,2023,1.92,6.0,20
2,2024,0.00,0.0,0


In [17]:
# Step 2-0: 현재 df에서 가능한 검증 자동 판별
print(df.columns.tolist())

need_vif_cols = [
    "민원스트레스점수_2022","소음스트레스점수_2022","진동스트레스점수_2022",
    "민원스트레스점수_2023","소음스트레스점수_2023","진동스트레스점수_2023",
    "민원스트레스점수_2024","소음스트레스점수_2024","진동스트레스점수_2024",
]
need_repro_cols = [
    "총점_orig_2022","총점_interp_2022",
    "총점_orig_2023","총점_interp_2023",
    "총점_orig_2024","총점_interp_2024",
]

missing_vif = [c for c in need_vif_cols if c not in df.columns]
missing_repro = [c for c in need_repro_cols if c not in df.columns]

print("VIF 누락:", missing_vif)
print("재현성 누락:", missing_repro)


['자치구', '총점_orig_2022', '총점_interp_2022', '순위_orig_2022', '순위_interp_2022', '점수차(orig-interp)_2022', '순위차(orig-interp)_2022', '절대순위차_2022', '총점_orig_2023', '총점_interp_2023', '순위_orig_2023', '순위_interp_2023', '점수차(orig-interp)_2023', '순위차(orig-interp)_2023', '절대순위차_2023', '총점_orig_2024', '총점_interp_2024', '순위_orig_2024', '순위_interp_2024', '점수차(orig-interp)_2024', '순위차(orig-interp)_2024', '절대순위차_2024']
VIF 누락: ['민원스트레스점수_2022', '소음스트레스점수_2022', '진동스트레스점수_2022', '민원스트레스점수_2023', '소음스트레스점수_2023', '진동스트레스점수_2023', '민원스트레스점수_2024', '소음스트레스점수_2024', '진동스트레스점수_2024']
재현성 누락: []


--------------------

## Step 2. 컬럼 매핑(여기만 맞추면 전체 코드 동작)

### 필수 매핑
- 자치구 컬럼
- 연도별 하위지표(민원/소음/진동)
- 원본 vs 보정(보간) 총점 컬럼 (재현성 검증용)


In [18]:
# 전제: Step 1에서 df 로드 완료
# 예)
# df = pd.read_csv(".../noise_rank_compare.csv", encoding="utf-8-sig")

import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression

gu_col = "자치구"

# VIF용 하위지표 (있을 경우)
feature_map = {
    2022: ["민원스트레스점수_2022", "소음스트레스점수_2022", "진동스트레스점수_2022"],
    2023: ["민원스트레스점수_2023", "소음스트레스점수_2023", "진동스트레스점수_2023"],
    2024: ["민원스트레스점수_2024", "소음스트레스점수_2024", "진동스트레스점수_2024"],
}

# 재현성용 원본 vs 보간 총점
score_pairs = {
    2022: ("총점_orig_2022", "총점_interp_2022"),
    2023: ("총점_orig_2023", "총점_interp_2023"),
    2024: ("총점_orig_2024", "총점_interp_2024"),
}

# 존재 체크
need_vif = [c for y in feature_map for c in feature_map[y]]
need_repro = [c for y in score_pairs for c in score_pairs[y]]

missing_vif = [c for c in need_vif if c not in df.columns]
missing_repro = [c for c in need_repro if c not in df.columns]

print("VIF 누락 컬럼:", missing_vif)
print("재현성 누락 컬럼:", missing_repro)

can_vif = (len(missing_vif) == 0)
can_repro = (len(missing_repro) == 0)

print("VIF 가능:", can_vif)
print("재현성 가능:", can_repro)


VIF 누락 컬럼: ['민원스트레스점수_2022', '소음스트레스점수_2022', '진동스트레스점수_2022', '민원스트레스점수_2023', '소음스트레스점수_2023', '진동스트레스점수_2023', '민원스트레스점수_2024', '소음스트레스점수_2024', '진동스트레스점수_2024']
재현성 누락 컬럼: []
VIF 가능: False
재현성 가능: True


## Step 3. VIF 계산 (가능한 경우만 실행)

### 주의
- 총점 컬럼은 VIF 입력에 넣지 않습니다.
- 하위지표(민원/소음/진동)만 사용합니다.


In [ ]:
# VIF 전용 파일 로드
vif_df = pd.read_csv("소음최종점수_2022_2024.csv",encoding="utf-8-sig")

feature_map_vif = {
    2022: ["민원스트레스점수_2022", "소음스트레스점수_2022", "진동스트레스점수_2022"],
    2023: ["민원스트레스점수_2023", "소음스트레스점수_2023", "진동스트레스점수_2023"],
    2024: ["민원스트레스점수_2024", "소음스트레스점수_2024", "진동스트레스점수_2024"],
}

vif_rows = []
for y, cols in feature_map_vif.items():
    v = compute_vif(vif_df[cols]).copy()
    v["연도"] = y
    v["판정"] = np.where(v["VIF"] >= 10, "높음(10+)",
                 np.where(v["VIF"] >= 5, "주의(5~10)", "양호(<5)"))
    vif_rows.append(v)

vif_result = pd.concat(vif_rows, ignore_index=True)[["연도","변수","VIF","표본수","판정"]]
display(vif_result)


,연도,변수,VIF,표본수,판정
0,2022,민원스트레스점수_2022,1.007706,25,양호(<5)
1,2022,진동스트레스점수_2022,1.006262,25,양호(<5)
2,2022,소음스트레스점수_2022,1.001440,25,양호(<5)
3,2023,소음스트레스점수_2023,1.181447,25,양호(<5)
4,2023,민원스트레스점수_2023,1.146004,25,양호(<5)
5,2023,진동스트레스점수_2023,1.033300,25,양호(<5)
6,2024,민원스트레스점수_2024,1.051915,25,양호(<5)
7,2024,소음스트레스점수_2024,1.050272,25,양호(<5)
8,2024,진동스트레스점수_2024,1.002731,25,양호(<5)


## Step 3-0. 보간후 파일 생성 (최소 버전)
- 원본 파일에서 이상치 월만 보간해서 `소음최종점수_보간후_2022_2024.csv` 생성
- DB 재조회 없이 현재 원본 파일 기준으로 빠르게 생성


In [ ]:
import pandas as pd
import numpy as np

orig = pd.read_csv("소음최종점수_2022_2024.csv", encoding="utf-8-sig")

display(orig.head())
print(orig.columns.tolist())


,자치구,민원스트레스점수_2022,민원스트레스점수_2023,민원스트레스점수_2024,소음스트레스점수_2022,소음스트레스점수_2023,소음스트레스점수_2024,진동스트레스점수_2022,진동스트레스점수_2023,진동스트레스점수_2024,소음원점수_2022,소음원점수_2023,소음원점수_2024,소음총점수_2022,소음총점수_2023,소음총점수_2024
0,종로구,6.09,6.98,6.52,3.36,3.20,3.52,3.96,4.16,4.73,13.41,14.34,14.77,13.97,14.94,15.39
1,중구,8.00,4.44,4.25,3.03,2.13,2.34,2.63,2.96,4.37,13.66,9.53,10.96,14.23,9.93,11.42
2,용산구,4.97,6.67,5.83,4.45,6.69,3.20,3.34,4.02,4.14,12.76,17.38,13.17,13.29,18.10,13.72
3,성동구,3.49,4.33,4.16,6.29,8.00,6.66,3.87,4.62,5.09,13.65,16.95,15.91,14.22,17.66,16.57
4,광진구,2.13,1.90,1.78,4.94,3.40,2.68,4.25,4.73,4.82,11.32,10.03,9.28,11.79,10.45,9.67


['자치구', '민원스트레스점수_2022', '민원스트레스점수_2023', '민원스트레스점수_2024', '소음스트레스점수_2022', '소음스트레스점수_2023', '소음스트레스점수_2024', '진동스트레스점수_2022', '진동스트레스점수_2023', '진동스트레스점수_2024', '소음원점수_2022', '소음원점수_2023', '소음원점수_2024', '소음총점수_2022', '소음총점수_2023', '소음총점수_2024']


## Step 3-1. 하위 점수 보간(연도축 2022~2024)

### 보간 대상
- 소음/진동/민원 스트레스 점수 컬럼 (연도별 3점)

### 방식
- 자치구별로 2022~2024 선형 보간
- 결측이 없으면 값 유지


In [26]:
interp = orig.copy()

groups = [
    ["민원스트레스점수_2022", "민원스트레스점수_2023", "민원스트레스점수_2024"],
    ["소음스트레스점수_2022", "소음스트레스점수_2023", "소음스트레스점수_2024"],
    ["진동스트레스점수_2022", "진동스트레스점수_2023", "진동스트레스점수_2024"],
]

for cols in groups:
    interp[cols] = interp[cols].apply(pd.to_numeric, errors="coerce")
    interp[cols] = interp[cols].T.interpolate(method="linear", limit_direction="both").T


## Step 3-2. 총점 재계산 (25점 환산)
- 총점 = (민원 + 소음 + 진동) * (25/24)
- 소수점 2자리 반올림


In [27]:
for y in [2022, 2023, 2024]:
    interp[f"소음총점수_{y}"] = (
        interp[f"민원스트레스점수_{y}"] +
        interp[f"소음스트레스점수_{y}"] +
        interp[f"진동스트레스점수_{y}"]
    ) * (25/24)
    interp[f"소음총점수_{y}"] = pd.to_numeric(interp[f"소음총점수_{y}"], errors="coerce").round(2)

display(interp[[c for c in interp.columns if "소음총점수_" in c]].head())


,소음총점수_2022,소음총점수_2023,소음총점수_2024
0,13.97,14.94,15.39
1,14.23,9.93,11.42
2,13.29,18.10,13.72
3,14.22,17.66,16.57
4,11.79,10.45,9.67


## Step 3-3. 보간후 파일 저장


In [ ]:

interp.to_csv("소음최종점수_보간후_2022_2024.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", interp_path)


저장 완료: C:/Users/yiho1/8ssible-Healing-Seoul-Analysis/프로젝트 HY/소음최종점수_보간후_2022_2024.csv


In [ ]:

from scipy.stats import pearsonr, spearmanr

# 파일 경로

orig = pd.read_csv("소음최종점수_2022_2024.csv", encoding="utf-8-sig")
interp = pd.read_csv("소음최종점수_보간후_2022_2024.csv", encoding="utf-8-sig")


## Step 4. 재현성 상관계수 (원본 vs 보간 총점)
- SciPy 없이 NumPy/Pandas로 Pearson, Spearman(순위변환 후 Pearson) 계산
- 결과를 CSV로 저장


In [ ]:
import os
import pandas as pd
import numpy as np

out_dir = r"C:/Users/yiho1/8ssible-Healing-Seoul-Analysis/프로젝트 HY"

orig = pd.read_csv("소음최종점수_2022_2024.csv", encoding="utf-8-sig")
interp = pd.read_csv("소음최종점수_보간후_2022_2024.csv", encoding="utf-8-sig")

# 총점 컬럼 없으면 생성
for d in [orig, interp]:
    for y in [2022, 2023, 2024]:
        tcol = f"소음총점수_{y}"
        if tcol not in d.columns:
            d[tcol] = (
                pd.to_numeric(d[f"민원스트레스점수_{y}"], errors="coerce") +
                pd.to_numeric(d[f"소음스트레스점수_{y}"], errors="coerce") +
                pd.to_numeric(d[f"진동스트레스점수_{y}"], errors="coerce")
            ) * (25/24)
        d[tcol] = pd.to_numeric(d[tcol], errors="coerce").round(2)

cmp = orig[["자치구","소음총점수_2022","소음총점수_2023","소음총점수_2024"]].merge(
    interp[["자치구","소음총점수_2022","소음총점수_2023","소음총점수_2024"]],
    on="자치구", suffixes=("_orig","_interp")
)

pairs = {
    2022: ("소음총점수_2022_orig","소음총점수_2022_interp"),
    2023: ("소음총점수_2023_orig","소음총점수_2023_interp"),
    2024: ("소음총점수_2024_orig","소음총점수_2024_interp"),
}

corr_rows = []
rank_rows = []
rank_detail_all = []

for y, (c1, c2) in pairs.items():
    a = pd.to_numeric(cmp[c1], errors="coerce")
    b = pd.to_numeric(cmp[c2], errors="coerce")
    m = a.notna() & b.notna()

    aa = a[m].to_numpy()
    bb = b[m].to_numpy()

    # Pearson
    pearson_r = float(np.corrcoef(aa, bb)[0,1]) if len(aa) > 1 else np.nan

    # Spearman = rank 변환 후 Pearson
    ra = pd.Series(aa).rank(method="average").to_numpy()
    rb = pd.Series(bb).rank(method="average").to_numpy()
    spearman_rho = float(np.corrcoef(ra, rb)[0,1]) if len(ra) > 1 else np.nan

    corr_rows.append({
        "연도": y,
        "표본수": int(m.sum()),
        "Pearson_r": round(pearson_r, 6) if pd.notna(pearson_r) else np.nan,
        "Spearman_rho": round(spearman_rho, 6) if pd.notna(spearman_rho) else np.nan,
        "평균점수차(orig-interp)": round(float((a[m]-b[m]).mean()), 6),
        "최대절대점수차": round(float((a[m]-b[m]).abs().max()), 6),
    })

    t = cmp[["자치구", c1, c2]].copy().dropna()
    t["순위_orig"] = t[c1].rank(ascending=False, method="min")
    t["순위_interp"] = t[c2].rank(ascending=False, method="min")
    t["순위차(orig-interp)"] = t["순위_orig"] - t["순위_interp"]
    t["절대순위차"] = t["순위차(orig-interp)"].abs()
    t["연도"] = y

    rank_rows.append({
        "연도": y,
        "표본수": len(t),
        "평균절대순위변화": round(float(t["절대순위차"].mean()), 4),
        "최대순위변화": round(float(t["절대순위차"].max()), 4),
        "순위변화발생개수": int((t["절대순위차"] > 0).sum()),
    })

    rank_detail_all.append(t[["자치구","연도",c1,c2,"순위_orig","순위_interp","순위차(orig-interp)","절대순위차"]])

repro_corr = pd.DataFrame(corr_rows).sort_values("연도")
rank_summary = pd.DataFrame(rank_rows).sort_values("연도")
rank_detail = pd.concat(rank_detail_all, ignore_index=True).sort_values(["연도","절대순위차"], ascending=[True,False])

display(repro_corr)
display(rank_summary)
display(rank_detail.head(30))

repro_corr.to_csv(os.path.join(out_dir, "validation_repro_corr.csv"), index=False, encoding="utf-8-sig")
rank_summary.to_csv(os.path.join(out_dir, "validation_rank_change_summary.csv"), index=False, encoding="utf-8-sig")
rank_detail.to_csv(os.path.join(out_dir, "validation_rank_change_detail.csv"), index=False, encoding="utf-8-sig")

print("저장 완료: validation_repro_corr.csv, validation_rank_change_summary.csv, validation_rank_change_detail.csv")


,연도,표본수,Pearson_r,Spearman_rho,평균점수차(orig-interp),최대절대점수차
0,2022,25,1.0,1.0,0.0,0.0
1,2023,25,1.0,1.0,0.0,0.0
2,2024,25,1.0,1.0,0.0,0.0


,연도,표본수,평균절대순위변화,최대순위변화,순위변화발생개수
0,2022,25,0.0,0.0,0
1,2023,25,0.0,0.0,0
2,2024,25,0.0,0.0,0


,자치구,연도,소음총점수_2022_orig,소음총점수_2022_interp,순위_orig,순위_interp,순위차(orig-interp),절대순위차,소음총점수_2023_orig,소음총점수_2023_interp,소음총점수_2024_orig,소음총점수_2024_interp
0,종로구,2022,13.97,13.97,10.0,10.0,0.0,0.0,NaN,NaN,NaN,NaN
1,중구,2022,14.23,14.23,8.0,8.0,0.0,0.0,NaN,NaN,NaN,NaN
2,용산구,2022,13.29,13.29,11.0,11.0,0.0,0.0,NaN,NaN,NaN,NaN
3,성동구,2022,14.22,14.22,9.0,9.0,0.0,0.0,NaN,NaN,NaN,NaN
4,광진구,2022,11.79,11.79,14.0,14.0,0.0,0.0,NaN,NaN,NaN,NaN
5,동대문구,2022,14.41,14.41,6.0,6.0,0.0,0.0,NaN,NaN,NaN,NaN
6,중랑구,2022,9.14,9.14,23.0,23.0,0.0,0.0,NaN,NaN,NaN,NaN
7,성북구,2022,9.81,9.81,21.0,21.0,0.0,0.0,NaN,NaN,NaN,NaN
8,강북구,2022,11.08,11.08,15.0,15.0,0.0,0.0,NaN,NaN,NaN,NaN
9,도봉구,2022,9.65,9.65,22.0,22.0,0.0,0.0,NaN,NaN,NaN,NaN


저장 완료: validation_repro_corr.csv, validation_rank_change_summary.csv, validation_rank_change_detail.csv


## Step 5. 해석 가이드
- Pearson/Spearman가 1에 가까울수록 원본 vs 보간 점수 구조가 동일
- 평균점수차/최대절대점수차가 0에 가까울수록 보간 영향이 적음
- 순위변동이 0이면 정책 우선순위 해석은 동일


In [32]:
# 1) 파일이 실제로 다른지 먼저 확인
for y in [2022, 2023, 2024]:
    c = f"소음총점수_{y}"
    diff = (orig[c] - interp[c]).abs()
    print(y, "max_diff:", diff.max(), "changed_rows:", (diff > 0).sum())


2022 max_diff: 0.0 changed_rows: 0
2023 max_diff: 0.0 changed_rows: 0
2024 max_diff: 0.0 changed_rows: 0


In [33]:
# 2) 재현성 상관 + 순위변동
rows = []
for y in [2022, 2023, 2024]:
    c = f"소음총점수_{y}"
    a = orig[c].astype(float)
    b = interp[c].astype(float)

    pearson_r = a.corr(b, method="pearson")
    spearman_rho = a.corr(b, method="spearman")

    r1 = a.rank(ascending=False, method="min")
    r2 = b.rank(ascending=False, method="min")
    shift = (r1 - r2).abs()

    rows.append({
        "연도": y,
        "Pearson_r": pearson_r,
        "Spearman_rho": spearman_rho,
        "평균점수차": (a-b).mean(),
        "최대절대점수차": (a-b).abs().max(),
        "평균절대순위변화": shift.mean(),
        "최대순위변화": shift.max(),
        "순위변화개수": (shift > 0).sum()
    })

recheck = pd.DataFrame(rows)
display(recheck)


,연도,Pearson_r,Spearman_rho,평균점수차,최대절대점수차,평균절대순위변화,최대순위변화,순위변화개수
0,2022,1.0,1.0,0.0,0.0,0.0,0.0,0
1,2023,1.0,1.0,0.0,0.0,0.0,0.0,0
2,2024,1.0,1.0,0.0,0.0,0.0,0.0,0
